Federated Learning

In [ ]:
import tensorflow as tf
import numpy as np

# 1. Build Model
def create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(1, input_shape=(2,), activation='linear')
    ])

    model.compile(
        optimizer='sgd',
        loss='mean_squared_error'
    )

    return model


# 2. Initialize Global Model
global_model = create_model()


# 3. Client Data (Local)
X_A = np.array([[1.0, 2.0], [2.0, 3.0]])
y_A = np.array([3.0, 5.0])

X_B = np.array([[3.0, 4.0], [5.0, 6.0]])
y_B = np.array([7.0, 11.0])


# 4. Train Client A
client_A = create_model()
client_A.set_weights(global_model.get_weights())

client_A.fit(X_A, y_A, epochs=5)
weights_A = client_A.get_weights()


# 5. Train Client B
client_B = create_model()
client_B.set_weights(global_model.get_weights())

client_B.fit(X_B, y_B, epochs=5)
weights_B = client_B.get_weights()


# 6. Federated Averaging (FedAvg)
new_weights = []
for wA, wB in zip(weights_A, weights_B):
    avg = (wA + wB) / 2
    new_weights.append(avg)


# 7. Update Global Model
global_model.set_weights(new_weights)


# 8. Display Results
print("Updated Global Weights:")
print(global_model.get_weights()[0])

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 382ms/step - loss: 6.0260
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step - loss: 3.8779
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - loss: 2.4980
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - loss: 1.6114
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - loss: 1.0419
Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 361ms/step - loss: 32.8688
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - loss: 0.4914
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - loss: 0.0145
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - loss: 0.0075
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - loss: 0.0073
Updated Global Weights:
[[0.5515473]
 [1.1045799]]


Adaptive Feature Fusion

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

# Input features
f1 = keras.Input(shape=(2,))
f2 = keras.Input(shape=(2,))

# Combine features
combined = keras.layers.Concatenate()([f1, f2])

# Generate attention weights (2 values)
weights = keras.layers.Dense(2, activation='softmax')(combined)

# Split weights
w1 = weights[:, 0:1]
w2 = weights[:, 1:2]

# Apply weights
f1_weighted = keras.layers.Multiply()([f1, w1])
f2_weighted = keras.layers.Multiply()([f2, w2])

# Final fusion
output = keras.layers.Add()([f1_weighted, f2_weighted])

# Model
model = keras.Model(inputs=[f1, f2], outputs=output)

# Dummy data
f1_data = np.array([[1.0, 0.5]])
f2_data = np.array([[0.2, 0.8]])

# Prediction
result = model.predict([f1_data, f2_data], verbose=0)

print("Feature 1:", f1_data[0])
print("Feature 2:", f2_data[0])
print("Fused Output:", result[0])

Feature 1: [1.  0.5]
Feature 2: [0.2 0.8]
Fused Output: [0.35023618 0.7436614 ]


Octree Segmentation

In [ ]:
import numpy as np

# Node
class OctreeNode:
    def __init__(self, center, size, points, depth=0):
        self.center = center
        self.size = size
        self.points = points
        self.depth = depth
        self.children = []


# Build tree
def build(node, max_depth, max_points):

    if node.depth >= max_depth or len(node.points) <= max_points:
        return node

    half = node.size / 2
    quarter = node.size / 4

    offsets = [
        (1,1,1), (1,1,-1), (1,-1,1), (1,-1,-1),
        (-1,1,1), (-1,1,-1), (-1,-1,1), (-1,-1,-1)
    ]

    for dx, dy, dz in offsets:

        new_center = node.center + np.array([dx, dy, dz]) * quarter

        cond_x = (node.points[:,0] >= node.center[0]) if dx > 0 else (node.points[:,0] < node.center[0])
        cond_y = (node.points[:,1] >= node.center[1]) if dy > 0 else (node.points[:,1] < node.center[1])
        cond_z = (node.points[:,2] >= node.center[2]) if dz > 0 else (node.points[:,2] < node.center[2])

        child_pts = node.points[cond_x & cond_y & cond_z]

        if len(child_pts) > 0:
            child = OctreeNode(new_center, half, child_pts, node.depth + 1)
            node.children.append(build(child, max_depth, max_points))

    return node


# Print tree (IMPORTANT for understanding)
def print_tree(node):
    print("Depth:", node.depth,
          "| Points:", len(node.points),
          "| Children:", len(node.children))

    for child in node.children:
        print_tree(child)


# -------- Simulation --------

np.random.seed(42)
points = np.random.uniform(0, 100, (1000, 3))

center = np.array([50.0, 50.0, 50.0])
size = 100.0

root = OctreeNode(center, size, points)

print("Building Octree...")
tree = build(root, max_depth=3, max_points=50)
print("Done!\n")

# Show structure
print_tree(tree)

Building Octree...
Done!

Depth: 0 | Points: 1000 | Children: 8
Depth: 1 | Points: 129 | Children: 8
Depth: 2 | Points: 19 | Children: 0
Depth: 2 | Points: 21 | Children: 0
Depth: 2 | Points: 15 | Children: 0
Depth: 2 | Points: 20 | Children: 0
Depth: 2 | Points: 13 | Children: 0
Depth: 2 | Points: 15 | Children: 0
Depth: 2 | Points: 13 | Children: 0
Depth: 2 | Points: 13 | Children: 0
Depth: 1 | Points: 132 | Children: 8
Depth: 2 | Points: 16 | Children: 0
Depth: 2 | Points: 11 | Children: 0
Depth: 2 | Points: 16 | Children: 0
Depth: 2 | Points: 18 | Children: 0
Depth: 2 | Points: 16 | Children: 0
Depth: 2 | Points: 23 | Children: 0
Depth: 2 | Points: 15 | Children: 0
Depth: 2 | Points: 17 | Children: 0
Depth: 1 | Points: 126 | Children: 8
Depth: 2 | Points: 13 | Children: 0
Depth: 2 | Points: 15 | Children: 0
Depth: 2 | Points: 18 | Children: 0
Depth: 2 | Points: 17 | Children: 0
Depth: 2 | Points: 17 | Children: 0
Depth: 2 | Points: 18 | Children: 0
Depth: 2 | Points: 17 | Children:

Discrete Wavelet Transform (DWT)

In [ ]:
#Python · PyWavelets
import numpy as np
import pywt

# 1. Define the input signal
signal = np.array([1, 2, 3, 4])
print("Original Signal:", signal)

# 2. Apply 1D DWT using Haar wavelet
#    cA = Approximation Coefficients (Low frequency)
#    cD = Detail Coefficients (High frequency)
cA, cD = pywt.dwt(signal, 'haar')

print("\n--- Level 1 DWT Coefficients ---")
print(f"Approximation (cA): {np.round(cA, 3)}")  # [2.121, 4.950]
print(f"Detail       (cD): {np.round(cD, 3)}")  # [-0.707, -0.707]

# 3. Reconstruction (Inverse DWT) — proves perfect reconstruction
reconstructed = pywt.idwt(cA, cD, 'haar')
print(f"\nReconstructed Signal: {np.round(reconstructed, 1)}")  # [1, 2, 3, 4]

Original Signal: [1 2 3 4]

--- Level 1 DWT Coefficients ---
Approximation (cA): [2.121 4.95 ]
Detail       (cD): [-0.707 -0.707]

Reconstructed Signal: [1. 2. 3. 4.]


Graph Attention Networks (GAT)

In [ ]:
import tensorflow as tf
import numpy as np

# Node features (A, B, C)
h = tf.constant([
    [1.0, 1.0],   # A
    [1.0, 2.0],   # B
    [3.0, 4.0]    # C
])

# Linear transformation (W * h)
W = tf.keras.layers.Dense(2, use_bias=False)
z = W(h)

# Raw attention scores (A -> B, C)
scores = tf.constant([1.5, 0.1])

# Softmax (normalize)
alpha = tf.nn.softmax(scores)
print("Attention Weights:", alpha.numpy())

# Take neighbor features (B, C)
neighbors = z[1:]

# Weighted aggregation
alpha = tf.reshape(alpha, (-1, 1))
new_A = tf.reduce_sum(alpha * neighbors, axis=0)

# Activation (ELU)
final_A = tf.nn.elu(new_A)

print("New Representation for Node A:", final_A.numpy())

Attention Weights: [0.8021839  0.19781613]
New Representation for Node A: [ 1.2376845 -0.8200085]
